<font size="10" color="DarkSlateGray"><b>Ultralytics Object Detection</b></font>

<img src="https://drive.google.com/uc?export=view&id=1u71bvuqnuzD8EgT6Jqq6frWwM7W1PnHP" width="1000">

In [1]:
# Install YOLO and Roboflow
!pip install -qq ultralytics==8.3.119 roboflow >/dev/null 2>&1

In [2]:
import ultralytics
from roboflow import Roboflow
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## YOLO11 Model Guide

<img src="https://cdn.jsdelivr.net/gh/ultralytics/assets@main/docs/ultralytics-yolov8-tasks-banner.avif
" width="1000">

YOLO11 supports multiple computer vision tasks. Choose a task type and model size based on your needs.

---

### Task Types

**Detection** `yolo11`  
Locates and draws bounding boxes around objects in images

**Segmentation** `yolo11-seg`  
Identifies precise pixel-level object boundaries and shapes

**Classification** `yolo11-cls`  
Assigns a single category label to an entire image

**Pose Estimation** `yolo11-pose`  
Detects human body keypoints for pose and skeleton tracking

**Oriented Detection** `yolo11-obb`  
Detects objects with rotated bounding boxes (non-axis-aligned)

---

### Model Sizes

**n (Nano)** — Fastest, basic accuracy → Mobile and edge devices  
**s (Small)** — Fast, good accuracy → Real-time applications  
**m (Medium)** — Moderate speed, better accuracy → Balanced performance  
**l (Large)** — Slow, high accuracy → Accuracy over speed  
**x (Extra)** — Slowest, highest accuracy → Maximum accuracy requirements

---

### Usage

Combine task prefix and size to load your model:
```python
from ultralytics import YOLO

# Format: {task}{size}.pt
model = YOLO("yolo11s.pt")          # Small detection
model = YOLO("yolo11m-seg.pt")      # Medium segmentation
model = YOLO("yolo11l-pose.pt")     # Large pose estimation
```

**Naming pattern:** `{task}{size}.pt` (e.g., `yolo11m-seg.pt`)

## Connections

In [4]:
import os
from google.colab import drive, userdata
from pathlib import Path

**Connect to Google Drive**

In [5]:
# Connect to Google Drive
if not Path("/content/drive").exists():
  drive.mount("/content/drive")

# Update 'base_dir' with the path to your private workspace on Google Drive
base_dir = Path("/content/drive/MyDrive/00_Workspace")

if not base_dir.exists():
  raise FileNotFoundError(f"Base directory {base_dir} does not exist")

# Define model directory
model_dir = base_dir / "models"

Mounted at /content/drive


## Download Dataset from Roboflow

In [6]:
ROBOFLOW_KEY = userdata.get('ROBOFLOW_KEY')

HOME = '/content'
!mkdir {HOME}/datasets
%cd {HOME}/datasets

workspace_id = "calpoly"
project_id = "traffic_signs-ik0wn"
VERSION = 7

rf = Roboflow(api_key=ROBOFLOW_KEY)
project = rf.workspace(workspace_id).project(project_id)
version = project.version(VERSION)
dataset = version.download("yolov11")

/content/datasets
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to traffic_signs-7 in yolov11:: 100%|██████████| 623/623 [00:00<00:00, 6521.04it/s]


## Train Model

In [ ]:
# Load a pretrained YOLOv11 segmentation model (small, good trade-off)
model = YOLO("yolo11s-seg.pt")

# Train the model on your Roboflow-exported dataset
# - data.yaml comes from Roboflow
# - epochs controls how long the model trains
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=120,
    project=model_dir
)

# Run validation on the validation split
# Prints mAP, precision, recall, and saves plots
model.val()


## Evaluation

In [ ]:
# Print all files/folders (choose your run name from here, e.g., "train", "train1", "train2", etc.)
for item in model_dir.iterdir():
    print(item)

In [13]:
# Set run name (training folder)
run_name = "train"

# Build path to best model weights
model_path = model_dir / run_name / "weights" / "best.pt"

# Load YOLO model
model = YOLO(model_path)

In [ ]:
# Path to test images folder
test_dir = Path(f"{dataset.location}/test/images")

# Get all image file paths
im_paths = [fpath for fpath in test_dir.iterdir()]

# Load images with OpenCV
images = [cv2.imread(path) for path in im_paths]

In [ ]:
# Run predictions on first 10 images and display results
for img in images[:10]:
    results = model(img)                  # Predict on image
    annotated = results[0].plot()         # Draw detections
    plt.imshow(annotated)                 # Show image
    plt.axis("off")                       # Hide axes
    plt.show()